In [1]:
import os
import csv
import time
import json
import math
import inspect
import itertools
from dataclasses import dataclass, asdict
import numpy as np
import gc
import wandb
import pickle

import torch
import torch.nn as nn
from torch.nn import functional as F

# =======================================================================
# EXECUTION MODE TOGGLE
# Set to True to do a 50-step test of all ablations. 
# Set to False to do the full 5000-step, 8-hour run.
# =======================================================================
IS_DUMMY_RUN = False

# --- CENTRAL CONFIGURATION ---
@dataclass
class ExperimentConfig:
    run_name: str = "baseline"
    seed: int = 1337
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # Wandb Logging Config
    wandb_log: bool = True                   # <--- WANDB TOGGLE
    wandb_project: str = "nanogpt-ablations" # <--- WANDB PROJECT NAME
    
    # Standard Hyperparameters (Matching original nanoGPT train_shakespeare_char.py)
    batch_size: int = 64
    block_size: int = 256
    max_iters: int = 5000 if not IS_DUMMY_RUN else 50
    eval_interval: int = 250 if not IS_DUMMY_RUN else 25
    eval_iters: int = 200 if not IS_DUMMY_RUN else 10
    learning_rate: float = 1e-3
    min_lr: float = 1e-4        
    warmup_iters: int = 100 if not IS_DUMMY_RUN else 10     
    lr_decay_iters: int = 5000 if not IS_DUMMY_RUN else 50  
    weight_decay: float = 1e-1
    grad_clip: float = 1.0
    vocab_size: int = 65
    bias: bool = False            # <--- Note: Kept as False to match NanoGPT shakespeare, but now dynamically applied

    log_interval: int = 10 

    # Model Architecture
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384
    dropout: float = 0.2
    
    # --- ABLATION FLAGS ---
    norm_type: str = "layernorm"        # Options: 'layernorm', 'rmsnorm, 'none'
    norm_placement: str = "pre"         # Options: 'pre', 'post'
    linear_type: str = "standard"       # Options: 'standard'
    pos_encoding: str = "learned"       # Options: 'learned', 'rope', 'alibi'
    residual_type: str = "standard"     # Options: 'standard', 'scaled', 'none'
    activation_type: str = "gelu"

In [2]:
#THIS IS WHERE NEW ARCHITECTURES GO ***HEREEEEE***
# 1. Modular Normalization Builder
def get_norm_layer(config, ndim):
    if config.norm_type == "layernorm":
        return nn.LayerNorm(ndim, bias=config.bias)
    elif config.norm_type == "rmsnorm":
        return nn.RMSNorm(ndim)
    elif config.norm_type == "none":
        return nn.Identity() #does absolutely nothing
    else:
        raise ValueError(f"Unknown norm_type: {config.norm_type}")

# 2. Modular Linear Builder
def get_linear_layer(config, in_features, out_features):
    if config.linear_type == "standard":
        return nn.Linear(in_features, out_features, bias=config.bias)
    elif config.linear_type == "ternary":
        return TernaryLinear(in_features, out_features, bias=config.bias)
    else:
        raise ValueError(f"Unknown linear_type: {config.linear_type}")

In [3]:
class TernaryLinear(nn.Module):
    """
    Implements Ternary Weights (-1, 0, 1) 
    using the Straight-Through Estimator (STE) trick.
    """
    def __init__(self, in_features, out_features, bias=False):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.normal(0, 0.02, size=(out_features, in_features)))
        if bias:
            self.bias = nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)

    def forward(self, x):
        gamma = self.weight.abs().mean().clamp(min=1e-8)
        w_scaled = self.weight / gamma
        w_rounded = torch.clamp(torch.round(w_scaled), -1.0, 1.0)
        
        # FIXED: Scale the rounded weights back up by gamma!
        w_quant = w_rounded * gamma
        
        # STE Trick
        w_ternary = (w_quant - self.weight).detach() + self.weight
        return F.linear(x, w_ternary, self.bias)

In [4]:
# helper functions for RoPE & ALiBi
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    """
    Precompute the frequency complex numbers for RoPE.
    dim: head dimension (n_embd // n_head)
    end: max sequence length (block_size)
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()  # (end, dim//2)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)  # complex64
    return freqs_cis

def rotate_half(x):
    """Rotates half the hidden dimensions."""
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb(xq, xk, freqs_cis):
    # xq, xk: (B, H, T, D)
    # freqs_cis: (T, D//2)
    B, H, T, D = xq.shape
    # reshape to (B, H, T, D//2, 2)
    xq_ = xq.float().reshape(B, H, T, D//2, 2)
    xk_ = xk.float().reshape(B, H, T, D//2, 2)
    # view as complex
    xq_complex = torch.view_as_complex(xq_)
    xk_complex = torch.view_as_complex(xk_)
    # expand freqs_cis to (1, 1, T, D//2)
    freqs_cis = freqs_cis.unsqueeze(0).unsqueeze(0)  # (1, 1, T, D//2)
    # multiply
    xq_out = torch.view_as_real(xq_complex * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_complex * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

In [5]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        assert config.pos_encoding in ['learned', 'rope', 'alibi', 'none'], f"Invalid PE: {config.pos_encoding}"
        
        self.c_attn = get_linear_layer(config, config.n_embd, 3 * config.n_embd)
        self.c_proj = get_linear_layer(config, config.n_embd, config.n_embd)
        
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        self.head_dim = config.n_embd // config.n_head
        
        self.pos_encoding = config.pos_encoding
        
        if self.pos_encoding == 'rope':
            assert self.head_dim % 2 == 0, "RoPE requires even head dimensions"
            self.register_buffer("freqs_cis", precompute_freqs_cis(self.head_dim, config.block_size))
        
        if self.pos_encoding == 'alibi':
            slopes = torch.tensor([2 ** (-(i + 1) * 8.0 / config.n_head) for i in range(config.n_head)])
            self.register_buffer("alibi_slopes", slopes.view(1, config.n_head, 1, 1))
        
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                     .view(1, 1, config.block_size, config.block_size))
    def forward(self, x):
            B, T, C = x.size()
            q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
            q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2) 
            k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
            
            if self.pos_encoding == 'rope':
                freqs_cis = self.freqs_cis[:T] 
                q, k = apply_rotary_emb(q, k, freqs_cis)
            
            # FIXED: Disable flash attention explicitly if using ALiBi
            use_flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention') and self.pos_encoding != 'alibi'
            
            if use_flash:
                y = torch.nn.functional.scaled_dot_product_attention(
                    q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True
                )
            else:
                att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
                att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
                
                if self.pos_encoding == 'alibi':
                    position_ids = torch.arange(T, device=x.device)
                    distance = position_ids[None, :] - position_ids[:, None]  
                    alibi_bias = self.alibi_slopes * distance.unsqueeze(0).to(att.dtype)
                    att = att + alibi_bias # Now 'att' safely exists!
                    
                att = F.softmax(att, dim=-1)
                att = self.attn_dropout(att)
                y = att @ v
            
            y = y.transpose(1, 2).contiguous().view(B, T, C)
            y = self.resid_dropout(self.c_proj(y))
            return y
        
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        if config.activation_type == "swiglu":
            hidden_dim = int((8/3)*config.n_embd)
            self.w1 = get_linear_layer(config, config.n_embd, hidden_dim)
            self.w2 = get_linear_layer(config, config.n_embd, hidden_dim) # ADDED
            self.w3 = get_linear_layer(config, hidden_dim, config.n_embd)
            self.silu = nn.SiLU()
        else:
            self.c_fc   = get_linear_layer(config, config.n_embd, 4 * config.n_embd)
            # DYNAMIC ACTIVATION
            self.act    = nn.ReLU() if config.activation_type == "relu" else nn.GELU()
            self.c_proj = get_linear_layer(config, 4 * config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        if self.config.activation_type == "swiglu":
            gate = self.silu(self.w1(x))
            data = self.w2(x) # FIXED: Now uses w2 for the linear pathway
            x = self.w3(gate * data)
        else:
            x = self.c_fc(x)
            x = self.act(x)   # FIXED: Respects ReLU if requested
            x = self.c_proj(x)
        return self.dropout(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.ln_1 = get_norm_layer(config, config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = get_norm_layer(config, config.n_embd)
        self.mlp = MLP(config)

        if self.config.residual_type == "scaled":
            self.res_scale = 1/ math.sqrt(self.config.n_layer)
        else:
            self.res_scale = 1

    def forward(self, x):
        if self.config.norm_placement == "post":
            if self.config.residual_type == "none":
                x = self.ln_1(self.attn(x))
                x = self.ln_2(self.mlp(x))
            else:
                x = self.ln_1(x + self.res_scale*self.attn(x))
                x = self.ln_2(x + self.res_scale*self.mlp(x))
        else:
            if self.config.residual_type == "none":
                x = self.attn(self.ln_1(x))
                x = self.mlp(self.ln_2(x))
            else:
                x = x + self.res_scale*self.attn(self.ln_1(x))
                x = x + self.res_scale*self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.wte = nn.Embedding(config.vocab_size, config.n_embd)
        
        # Only learned positional embeddings are used as a separate embedding
        self.wpe = nn.Embedding(config.block_size, config.n_embd) if config.pos_encoding == 'learned' else None
        
        self.drop = nn.Dropout(config.dropout)
        self.h = nn.ModuleList([Block(config) for _ in range(config.n_layer)])
        self.ln_f = get_norm_layer(config, config.n_embd)
        
        # <--- CHANGED: lm_head MUST stay full precision nn.Linear to protect token prob distributions.
        # It cannot be a get_linear_layer because weight tying with wte (Embeddings) requires full precision.
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=config.bias)
        
        # weight tying
        self.wte.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight') or pn.endswith('w3.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))
                
    def _init_weights(self, module):
        # Applies to both nn.Linear and our custom TernaryLinear
        if isinstance(module, (nn.Linear, TernaryLinear)):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if getattr(module, 'bias', None) is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def configure_optimizers(self, weight_decay, learning_rate, betas, device_type):
        """Splits weights so only 2D matrices get weight decay (Like the original nanoGPT)"""
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
        nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params, 'weight_decay': weight_decay},
            {'params': nodecay_params, 'weight_decay': 0.0}
        ]
        use_fused = 'fused' in inspect.signature(torch.optim.AdamW).parameters and device_type == 'cuda'
        extra_args = dict(fused=True) if use_fused else dict()
        return torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas, **extra_args)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        tok_emb = self.wte(idx)
        
        if self.wpe is not None:
            # learned absolute positional embeddings
            pos = torch.arange(0, t, dtype=torch.long, device=device)
            pos_emb = self.wpe(pos)
            x = self.drop(tok_emb + pos_emb)
        else:
            # RoPE or ALiBi: no addition here
            x = self.drop(tok_emb)
        
        for block in self.h:
            x = block(x)
                    
        # ONLY apply final layernorm in Pre-LN designs. (Post-LN is already normalized)
        if self.config.norm_placement == "pre":
            x = self.ln_f(x)

        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1
            )
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None

        return logits, loss
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Generates sequence tokens for testing out the model"""
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [6]:
# --- LOGGING & RESUME HELPERS ---
def has_run_completed(run_name, filepath="metrics.csv"):
    """Checks if a run is already in the CSV so we can resume smoothly."""
    if not os.path.isfile(filepath):
        return False
    with open(filepath, mode='r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('run_name') == run_name:
                return True
    return False

def log_metrics_to_csv(config, metrics, filepath="metrics.csv"):
    row_data = {**asdict(config), **metrics}
    file_exists = os.path.isfile(filepath)
    with open(filepath, mode='a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row_data.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_data)

# --- MAIN TRAINING LOOP ---
def train_run(config: ExperimentConfig):
    print(f"\n{'='*50}\nStarting Run: {config.run_name}\n{'='*50}")
    
    # <--- WANDB INIT
    if config.wandb_log:
        wandb.init(
            project=config.wandb_project, 
            name=config.run_name, 
            config=asdict(config),
            tags=["dummy" if IS_DUMMY_RUN else "full"] # <--- Tags help you filter in WandB dashboard
        )
    
    torch.manual_seed(config.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(config.seed)

    # Autocast setup
    device_type = 'cuda' if 'cuda' in config.device else 'cpu'
    
    if device_type == 'cuda' and torch.cuda.is_bf16_supported():
        ptdtype = torch.bfloat16
    else:
        ptdtype = torch.float16 if device_type == 'cuda' else torch.float32

    # Data Loading
    train_data = np.memmap('data/train.bin', dtype=np.uint16, mode='r')
    val_data = np.memmap('data/val.bin', dtype=np.uint16, mode='r')

    def get_batch(split):
        data = train_data if split == 'train' else val_data
        ix = torch.randint(len(data) - config.block_size, (config.batch_size,))
        x = torch.stack([torch.from_numpy((data[i:i+config.block_size]).astype(np.int64)) for i in ix])
        y = torch.stack([torch.from_numpy((data[i+1:i+1+config.block_size]).astype(np.int64)) for i in ix])
        
        if device_type == 'cuda':
            x, y = x.pin_memory().to(config.device, non_blocking=True), y.pin_memory().to(config.device, non_blocking=True)
        else:
            x, y = x.to(config.device), y.to(config.device)
        return x, y

    # Model Setup
    model = GPT(config).to(config.device)
    optimizer = model.configure_optimizers(config.weight_decay, config.learning_rate, (0.9, 0.95), device_type)
    
    scaler = torch.amp.GradScaler(device_type, enabled=(ptdtype == torch.float16))

    n_params = sum(p.numel() for p in model.parameters())
    if config.wandb_log:
        wandb.config.update({"n_params": n_params})

    @torch.no_grad()
    def estimate_loss():
        out = {}
        model.eval()
        for split in ['train', 'val']:
            losses = torch.zeros(config.eval_iters)
            for k in range(config.eval_iters):
                X, Y = get_batch(split)
                with torch.autocast(device_type=device_type, dtype=ptdtype):
                    _, loss = model(X, Y)
                losses[k] = loss.item()
            out[split] = losses.mean().item()
        model.train()
        return out

    # Cosine Learning Rate Schedule
    def get_lr(it):
        if it < config.warmup_iters:
            return config.learning_rate * (it + 1) / (config.warmup_iters + 1)
        if it > config.lr_decay_iters:
            return config.min_lr
        decay_ratio = (it - config.warmup_iters) / (config.lr_decay_iters - config.warmup_iters)
        coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
        return config.min_lr + coeff * (config.learning_rate - config.min_lr)

    # Training Loop
    start_time = time.time()
    loss_history = {'train': [], 'val': [], 'iter': []}

    for iter_num in range(config.max_iters + 1):
        lr = get_lr(iter_num)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # Evaluate
        if iter_num % config.eval_interval == 0:
            losses = estimate_loss()
            print(f"Step {iter_num:4d} | Train Loss: {losses['train']:.4f} | Val Loss: {losses['val']:.4f} | LR: {lr:.4e}")
            loss_history['iter'].append(iter_num)
            loss_history['train'].append(losses['train'])
            loss_history['val'].append(losses['val'])
            
            # <--- WANDB LOGGING (EVAL)
            if config.wandb_log:
                wandb.log({
                    "iter": iter_num,
                    "val/loss": losses['val'],
                    "train/loss": losses['train'],
                    "lr": lr
                })

        # Forward & Backward Pass
        X, Y = get_batch('train')
        with torch.autocast(device_type=device_type, dtype=ptdtype):
            logits, loss = model(X, Y)

        scaler.scale(loss).backward()

        if config.grad_clip != 0.0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)

        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        
        if iter_num % config.log_interval == 0:
            print(f"iter {iter_num}: loss {loss.item():.4f}")
            # <--- WANDB LOGGING (TRAIN STEP)
            if config.wandb_log:
                wandb.log({
                    "iter": iter_num,
                    "train/step_loss": loss.item()
                })
            
    train_time = round(time.time() - start_time, 2)
        
    # --- TEXT GENERATION STEP (Saved to TXT, NOT printed) ---
    model.eval()
    try:
        with open('data/meta.pkl', 'rb') as f:
            meta = pickle.load(f)
        stoi, itos = meta['stoi'], meta['itos']
        context = torch.tensor([[stoi['\n']]], dtype=torch.long, device=config.device)
        generated_ids = model.generate(context, max_new_tokens=150)[0].tolist()
        generated_text = ''.join([itos[i] for i in generated_ids])
        
        # Save to a text file
        sample_path = f"output/{config.run_name}_sample.txt"
        with open(sample_path, "w", encoding="utf-8") as f:
            f.write(f"--- Model: {config.run_name} ---\n")
            f.write(f"--- Params: {n_params:,} ---\n\n") # <--- ADDED param count to txt
            f.write(generated_text)
            
        if config.wandb_log:
            wandb.log({"Sample Generation": wandb.Html(f"<pre>{generated_text}</pre>")})
    except Exception as e:
        print(f"Could not generate text: {e}")

    # Save Outputs
    torch.save(model.state_dict(), f"models/{config.run_name}.pt")
    with open(f"output/{config.run_name}_curves.json", 'w') as f:
        json.dump(loss_history, f)
        
    metrics = {
        "n_params": n_params,
        "final_train_loss": loss_history['train'][-1],
        "final_val_loss": loss_history['val'][-1],
        "training_time_sec": train_time
        
    }
    
    log_metrics_to_csv(config, metrics)
    print(f"✅ Finished {config.run_name} in {train_time}s. Saved to models/ and output/")
    
    # <--- WANDB FINISH (closes the run to prep for the next one)
    if config.wandb_log:
        wandb.finish()
    
    return model, metrics

In [7]:
# =======================================================================
# ABLATION EXPERIMENTS QUEUE
# =======================================================================
experiments = []

# BASELINE
experiments.append(ExperimentConfig(
    run_name="baseline", 
    pos_encoding="learned", norm_type="layernorm", norm_placement="pre",
    n_head=6, activation_type="gelu", residual_type="standard", linear_type="standard"
))

# AXIS A: Positional Encoding
experiments.append(ExperimentConfig(run_name="A1_no_pos_encoding", pos_encoding="none"))
experiments.append(ExperimentConfig(run_name="A2_rope", pos_encoding="rope"))
experiments.append(ExperimentConfig(run_name="A3_alibi", pos_encoding="alibi"))

# AXIS B: Normalization
experiments.append(ExperimentConfig(run_name="B1_rmsnorm", norm_type="rmsnorm"))
experiments.append(ExperimentConfig(run_name="B2_post_ln", norm_placement="post"))

# AXIS C: Attention Heads
experiments.append(ExperimentConfig(run_name="C1_1_head", n_head=1))
experiments.append(ExperimentConfig(run_name="C2_8_heads", n_head=8))
experiments.append(ExperimentConfig(run_name="C3_12_heads", n_head=12))

# AXIS D: Activation Functions
experiments.append(ExperimentConfig(run_name="D1_relu", activation_type="relu"))
experiments.append(ExperimentConfig(run_name="D2_swiglu", activation_type="swiglu"))

# AXIS E: Residual Connections
experiments.append(ExperimentConfig(run_name="E1_scaled_residual", residual_type="scaled"))
experiments.append(ExperimentConfig(run_name="E2_no_residuals", residual_type="none"))

# AXIS F: Context Length
experiments.append(ExperimentConfig(run_name="F1_context_128", block_size=128))
experiments.append(ExperimentConfig(run_name="F2_context_512", block_size=512))

# AXIS G: Quantization
experiments.append(ExperimentConfig(run_name="G1_ternary_weights", linear_type="ternary"))

# =======================================================================
# RUN LOOP
# =======================================================================
for cfg in experiments:
    # 1. Resume Logic (Check if already done)
    if has_run_completed(cfg.run_name):
        print(f"⏭️  Skipping {cfg.run_name} (Already found in metrics.csv)")
        continue

    # 2. Run the training
    model, metrics = train_run(cfg)
    
    # 3. Clean VRAM
    model.cpu()
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n🎉 ALL EXPERIMENTS COMPLETED SUCCESSFULLY! Check metrics.csv")

# Final Cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Madhav\_netrc.



Starting Run: baseline


wandb: Currently logged in as: madhavkrishnan747 (madhavkrishnan747-australian-national-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step    0 | Train Loss: 4.2873 | Val Loss: 4.2822 | LR: 9.9010e-06
iter 0: loss 4.2606
iter 10: loss 3.1518
iter 20: loss 2.7404
iter 30: loss 2.6201
iter 40: loss 2.5713
iter 50: loss 2.5219
iter 60: loss 2.5036
iter 70: loss 2.4943
iter 80: loss 2.4935
iter 90: loss 2.4667
iter 100: loss 2.4685
iter 110: loss 2.4529
iter 120: loss 2.4279
iter 130: loss 2.4125
iter 140: loss 2.4156
iter 150: loss 2.4183
iter 160: loss 2.3791
iter 170: loss 2.3911
iter 180: loss 2.3428
iter 190: loss 2.2911
iter 200: loss 2.2667
iter 210: loss 2.2011
iter 220: loss 2.1989
iter 230: loss 2.1269
iter 240: loss 2.1349
Step  250 | Train Loss: 2.0135 | Val Loss: 2.1103 | LR: 9.9792e-04
iter 250: loss 2.0755
iter 260: loss 2.0224
iter 270: loss 2.0111
iter 280: loss 2.0150
iter 290: loss 1.9453
iter 300: loss 1.9310
iter 310: loss 1.9060
iter 320: loss 1.8910
iter 330: loss 1.8502
iter 340: loss 1.8195
iter 350: loss 1.8563
iter 360: loss 1.7893
iter 370: loss 1.7549
iter 380: loss 1.7478
iter 390: loss 1.74

iter,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇███
lr,▁███▇▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▂
train/loss,█▄▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/step_loss,██▅▄▄▄▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂
iter,5000
lr,0.0001
train/loss,0.63387
train/step_loss,0.83705
val/loss,1.70554


...

iter,▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇██
lr,▁███▇▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▂
train/loss,█▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/step_loss,█▆▆▆▆▅▄▄▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▃▂▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂
iter,5000
lr,0.0001
train/loss,0.34199
train/step_loss,0.59766
val/loss,2.02496



Starting Run: G1_ternary_weights


Step    0 | Train Loss: 4.3593 | Val Loss: 4.3546 | LR: 9.9010e-06
iter 0: loss 4.3267
iter 10: loss 3.2235
iter 20: loss 2.7605
iter 30: loss 2.6315
iter 40: loss 2.5745
iter 50: loss 2.5281
iter 60: loss 2.5167
iter 70: loss 2.5063
iter 80: loss 2.5072
iter 90: loss 2.4727
iter 100: loss 2.4803
iter 110: loss 2.4704
iter 120: loss 2.4455
iter 130: loss 2.4392
iter 140: loss 2.4394
iter 150: loss 2.4495
iter 160: loss 2.4318
iter 170: loss 2.4169
iter 180: loss 2.4103
iter 190: loss 2.3993
iter 200: loss 2.3667
iter 210: loss 2.3283
iter 220: loss 2.3684
iter 230: loss 2.3190
iter 240: loss 2.3380
Step  250 | Train Loss: 2.2405 | Val Loss: 2.2803 | LR: 9.9792e-04
iter 250: loss 2.2965
iter 260: loss 2.2714
iter 270: loss 2.2601
iter 280: loss 2.2746
iter 290: loss 2.2346
iter 300: loss 2.2113
iter 310: loss 2.1996
iter 320: loss 2.1793
iter 330: loss 2.1621
iter 340: loss 2.1263
iter 350: loss 2.1570
iter 360: loss 2.1010
iter 370: loss 2.0864
iter 380: loss 2.0561
iter 390: loss 2.06

iter,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
lr,▁███▇▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▂
train/loss,█▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/step_loss,██▅▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val/loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
iter,5000
lr,0.0001
train/loss,1.04771
train/step_loss,1.13663
val/loss,1.48774



🎉 ALL EXPERIMENTS COMPLETED SUCCESSFULLY! Check metrics.csv
